In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

SOURCE_DIR = Path("/content/drive/MyDrive/CPTAC-LSCC")

print("Check TUMOR files:")
print(len(list((SOURCE_DIR / "TUMOR").glob("*.svs"))))

print("Check NORMAL files:")
print(len(list((SOURCE_DIR / "NORMAL").glob("*.svs"))))

Check TUMOR files:
694
Check NORMAL files:
387


In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

# =========================================================
# PATHS
# =========================================================
BASE_DIR = Path("/content/drive/MyDrive/CPTAC-LSCC")
SOURCE_DIR = BASE_DIR
OUTPUT_DIR = Path("/content/drive/MyDrive/CPTAC-LSCC_split_balanced")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# =========================================================
# HELPER: EXTRACT PATIENT ID
# Example:
# C3N-05929-23.svs -> C3N-05929
# =========================================================
def get_patient_id(filename: str) -> str:
    parts = filename.replace(".svs", "").split("-")
    return "-".join(parts[:2])

# =========================================================
# COLLECT FILES
# =========================================================
tumor_files = list((SOURCE_DIR / "TUMOR").glob("*.svs"))
normal_files = list((SOURCE_DIR / "NORMAL").glob("*.svs"))

print("Original slide counts")
print("---------------------")
print("Tumor :", len(tumor_files))
print("Normal:", len(normal_files))
print("Total :", len(tumor_files) + len(normal_files))

# =========================================================
# CHECK PATIENT OVERLAP BETWEEN TUMOR AND NORMAL
# =========================================================
tumor_patient_ids = set(get_patient_id(f.name) for f in tumor_files)
normal_patient_ids = set(get_patient_id(f.name) for f in normal_files)
both_patient_ids = tumor_patient_ids & normal_patient_ids

print("\nPatient overlap between classes")
print("-------------------------------")
print("Tumor patients :", len(tumor_patient_ids))
print("Normal patients:", len(normal_patient_ids))
print("Patients in BOTH classes:", len(both_patient_ids))

# =========================================================
# BUILD PATIENT -> FILES
# patient_to_files[pid] = list of (class_name, file_path)
# =========================================================
patient_to_files = defaultdict(list)

for f in tumor_files:
    pid = get_patient_id(f.name)
    patient_to_files[pid].append(("TUMOR", f))

for f in normal_files:
    pid = get_patient_id(f.name)
    patient_to_files[pid].append(("NORMAL", f))

all_patients = sorted(patient_to_files.keys())

# =========================================================
# PATIENT TYPE FOR STRATIFICATION
# TUMOR_ONLY / NORMAL_ONLY / BOTH
# =========================================================
patient_types = {}

for pid, items in patient_to_files.items():
    classes = set(cls for cls, _ in items)
    if classes == {"TUMOR"}:
        patient_types[pid] = "TUMOR_ONLY"
    elif classes == {"NORMAL"}:
        patient_types[pid] = "NORMAL_ONLY"
    else:
        patient_types[pid] = "BOTH"

stratify_labels = [patient_types[pid] for pid in all_patients]

print("\nPatient-level summary")
print("---------------------")
print("Total unique patients:", len(all_patients))
print("Patient type distribution:", Counter(stratify_labels))

# =========================================================
# PATIENT-LEVEL SPLIT: 70 / 15 / 15
# Use stratify only in first split
# =========================================================
train_patients, temp_patients, train_labels, temp_labels = train_test_split(
    all_patients,
    stratify_labels,
    train_size=0.70,
    stratify=stratify_labels,
    random_state=RANDOM_SEED
)

# IMPORTANT: no stratify here
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=RANDOM_SEED
)

print("\nPatient split sizes")
print("-------------------")
print("Train patients:", len(train_patients))
print("Val patients  :", len(val_patients))
print("Test patients :", len(test_patients))

# =========================================================
# VERIFY NO PATIENT LEAKAGE BEFORE MOVING
# =========================================================
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("\nLeakage check before moving")
print("---------------------------")
print("Train ∩ Val :", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test  :", len(val_set & test_set))

# =========================================================
# GATHER FILES PER SPLIT
# =========================================================
def gather_split_files(patient_ids):
    files = []
    for pid in patient_ids:
        files.extend(patient_to_files[pid])
    return files

train_files_all = gather_split_files(train_patients)
val_files_all = gather_split_files(val_patients)
test_files_all = gather_split_files(test_patients)

# =========================================================
# BALANCE EACH SPLIT AT SLIDE LEVEL
# Keep patient-level separation, then downsample larger class
# inside each split only. Extra slides go to "unused".
# =========================================================
def balance_split(files_with_class, split_name):
    tumor = [(cls, f) for cls, f in files_with_class if cls == "TUMOR"]
    normal = [(cls, f) for cls, f in files_with_class if cls == "NORMAL"]

    n_tumor = len(tumor)
    n_normal = len(normal)
    target = min(n_tumor, n_normal)

    random.shuffle(tumor)
    random.shuffle(normal)

    keep_tumor = tumor[:target]
    keep_normal = normal[:target]

    unused_tumor = tumor[target:]
    unused_normal = normal[target:]

    print(f"\n{split_name.upper()} before balancing -> TUMOR: {n_tumor}, NORMAL: {n_normal}")
    print(f"{split_name.upper()} after  balancing -> TUMOR: {len(keep_tumor)}, NORMAL: {len(keep_normal)}")

    keep = keep_tumor + keep_normal
    unused = unused_tumor + unused_normal
    return keep, unused

train_keep, train_unused = balance_split(train_files_all, "train")
val_keep, val_unused = balance_split(val_files_all, "val")
test_keep, test_unused = balance_split(test_files_all, "test")

# =========================================================
# CREATE OUTPUT FOLDERS
# =========================================================
for split in ["train", "val", "test", "unused"]:
    for cls in ["TUMOR", "NORMAL"]:
        os.makedirs(OUTPUT_DIR / split / cls, exist_ok=True)

# =========================================================
# MOVE FILES
# =========================================================
def move_files(files_with_class, split_name):
    moved = 0
    skipped = 0
    missing = 0
    errors = 0

    for cls, f in files_with_class:
        destination = OUTPUT_DIR / split_name / cls / f.name

        if destination.exists():
            skipped += 1
            continue

        if not f.exists():
            missing += 1
            print(f"Missing source file: {f}")
            continue

        try:
            shutil.move(str(f), str(destination))
            moved += 1
        except Exception as e:
            errors += 1
            print(f"Error moving {f.name}: {e}")

    print(f"{split_name.upper():6} -> moved: {moved}, skipped: {skipped}, missing: {missing}, errors: {errors}")

print("\nMoving selected balanced files")
print("------------------------------")
move_files(train_keep, "train")
move_files(val_keep, "val")
move_files(test_keep, "test")

print("\nMoving unused extra files")
print("-------------------------")
move_files(train_unused, "unused")
move_files(val_unused, "unused")
move_files(test_unused, "unused")

# =========================================================
# FINAL COUNTS
# =========================================================
def count_svs(folder: Path):
    return len(list(folder.glob("*.svs")))

print("\nFinal slide counts")
print("------------------")
for split in ["train", "val", "test", "unused"]:
    t = count_svs(OUTPUT_DIR / split / "TUMOR")
    n = count_svs(OUTPUT_DIR / split / "NORMAL")
    print(f"{split.upper():6} -> TUMOR: {t}, NORMAL: {n}, TOTAL: {t+n}")

# =========================================================
# FINAL PATIENT LEAKAGE CHECK ON TRAIN / VAL / TEST ONLY
# =========================================================
patient_locations = defaultdict(list)

for split in ["train", "val", "test"]:
    for cls in ["TUMOR", "NORMAL"]:
        folder = OUTPUT_DIR / split / cls
        for slide in folder.glob("*.svs"):
            pid = get_patient_id(slide.name)
            patient_locations[pid].append((split, cls, slide.name))

leakage = {
    pid: items
    for pid, items in patient_locations.items()
    if len(set(split for split, _, _ in items)) > 1
}

print("\nFinal leakage check")
print("-------------------")
print("Patients appearing in more than one split:", len(leakage))

if len(leakage) == 0:
    print("No patient leakage detected.")
else:
    for pid, items in sorted(leakage.items()):
        print(f"\n{pid}")
        for split, cls, fname in items:
            print(f"  [{split}][{cls}] {fname}")

print("\nDone.")
print("Saved to:", OUTPUT_DIR)

Original slide counts
---------------------
Tumor : 694
Normal: 387
Total : 1081

Patient overlap between classes
-------------------------------
Tumor patients : 212
Normal patients: 208
Patients in BOTH classes: 208

Patient-level summary
---------------------
Total unique patients: 212
Patient type distribution: Counter({'BOTH': 208, 'TUMOR_ONLY': 4})

Patient split sizes
-------------------
Train patients: 148
Val patients  : 32
Test patients : 32

Leakage check before moving
---------------------------
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

TRAIN before balancing -> TUMOR: 492, NORMAL: 274
TRAIN after  balancing -> TUMOR: 274, NORMAL: 274

VAL before balancing -> TUMOR: 105, NORMAL: 58
VAL after  balancing -> TUMOR: 58, NORMAL: 58

TEST before balancing -> TUMOR: 97, NORMAL: 55
TEST after  balancing -> TUMOR: 55, NORMAL: 55

Moving selected balanced files
------------------------------
TRAIN  -> moved: 548, skipped: 0, missing: 0, errors: 0
VAL    -> moved: 116, skipped:

In [ ]:
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/CPTAC-LSCC_split_balanced")

print("Slide counts per split")
print("----------------------")

for split in ["train", "val", "test"]:
    tumor = len(list((base_dir / split / "TUMOR").glob("*.svs")))
    normal = len(list((base_dir / split / "NORMAL").glob("*.svs")))
    total = tumor + normal
    print(f"{split.upper():5} -> TUMOR: {tumor}, NORMAL: {normal}, TOTAL: {total}")

Slide counts per split
----------------------
TRAIN -> TUMOR: 271, NORMAL: 271, TOTAL: 542
VAL   -> TUMOR: 58, NORMAL: 58, TOTAL: 116
TEST  -> TUMOR: 58, NORMAL: 58, TOTAL: 116
